# Modul Praktikum: Klasifikasi Teks & Analisis Sentimen
**Mata Kuliah Kecerdasan Artifisial**

Modul ini dirancang untuk mempelajari alur kerja (*pipeline*) klasifikasi teks secara menyeluruh menggunakan Python dan pustaka `scikit-learn`. Kita akan membangun model analisis sentimen untuk mengklasifikasikan ulasan e-commerce berbahasa Indonesia menjadi tiga kategori: **positif**, **negatif**, dan **netral**.

## Langkah 1: Pengunduhan Pustaka dan Persiapan Lingkungan
Memuat pustaka standar manipulasi data (`pandas`, `numpy`), alat pemrosesan teks (`nltk`), pembagian data, ekstraksi fitur TF-IDF, serta model klasifikasi `LogisticRegression`.

In [ ]:
# Menginstal pustaka NLTK untuk pemrosesan teks bahasa alami
!pip install nltk

import pandas as pd
import numpy as np
import re
import nltk
import string

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

# Mengunduh corpus stopwords dan tokenizer punkt dari NLTK
nltk.download('stopwords')
nltk.download('punkt')
print("Pustaka dan resource NLTK berhasil dimuat!")

## Langkah 2: Menyiapkan Dataset Ulasan Produk
Membuat dataset berisi ulasan produk e-commerce berbahasa Indonesia dengan sebaran label **positif**, **negatif**, dan **netral**.

In [ ]:
# Kamus data tiruan ulasan produk e-commerce beserta label sentimennya
data = {
    'text': [
        # Kategori Positif
        "Barangnya bagus banget, pengiriman cepat dan kurir ramah!",
        "Pengiriman lumayan lama, tapi kualitas produknya mantap pol.",
        "Kuenya enak banget, anak-anak di rumah pasti suka dan beli lagi.",
        "Pelayanan toko ramah dan pengirimannya kilat, top deh!",
        "Mantap jiwa, produk original dan berfungsi dengan sangat baik.",
        "Sangat puas belanja di sini, barang sesuai gambar dan berkualitas tinggi.",
        "Kualitas produk luar biasa, harga terjangkau, pengemasan aman.",
        "Recommended seller! Respon cepat dan barang mendarat dengan selamat.",
        
        # Kategori Negatif
        "Kecewa berat, barang datang dalam kondisi rusak dan tidak menyala.",
        "Sangat tidak rekomendasi! Penjual tidak responsif saat ditanya.",
        "Kualitas produk buruk, baru dipakai sehari langsung rusak.",
        "Penipu! Barang tidak pernah dikirim padahal sudah dibayar lunas.",
        "Barang palsu! Sangat kecewa, buang-buang uang saja.",
        "Kecewa sekali, packing hancur dan isinya tidak sesuai pesanan.",
        "Pelayanan buruk, chat dibalas lama dan ketus.",
        "Barang cacat dan penjual menolak untuk retur. Jangan beli di sini!",
        
        # Kategori Netral
        "Biasa saja sih, kualitas sesuai dengan harga yang murah.",
        "Bahan kainnya agak tipis tapi ya lumayanlah buat harga segini.",
        "Kurirnya nyasar lama banget, tapi barangnya aman sampai tujuan.",
        "Produk lumayan, tapi pengirimannya agak terlambat dari estimasi.",
        "Standar kualitasnya, tidak ada yang terlalu istimewa.",
        "Bungkusnya agak penyok sedikit tapi isinya aman."
    ],
    'sentiment': [
        "positif", "positif", "positif", "positif", "positif", "positif", "positif", "positif",
        "negatif", "negatif", "negatif", "negatif", "negatif", "negatif", "negatif", "negatif",
        "netral", "netral", "netral", "netral", "netral", "netral"
    ]
}

# Mengubah kamus menjadi pandas DataFrame
df = pd.DataFrame(data)
print(f"Total data ulasan: {len(df)} sampel")
df.head()

## Langkah 3: Prapemrosesan Teks (*Text Preprocessing*)
Membersihkan teks dengan melakukan *case folding* (huruf kecil), tokenisasi, penghapusan tanda baca, serta pembersihan kata umum (*stopwords*) bahasa Indonesia tanpa mengeliminasi kata negasi penting seperti **'tidak'**, **'kurang'**, **'jangan'**, dan **'bukan'**.

In [ ]:
# Mengambil daftar stopword bahasa Indonesia dari NLTK
stop_words = set(stopwords.words('indonesian'))

# Menentukan kata-kata negasi penting yang tidak boleh dibuang
important_words = {'tidak', 'kurang', 'jangan', 'bukan'}
stop_words = stop_words - important_words

def preprocess_text(text):
    # 1. Mengubah seluruh teks menjadi huruf kecil
    text = text.lower()
    # 2. Memecah kalimat menjadi token kata
    tokens = word_tokenize(text)
    # 3. Menyaring tanda baca dan stopword selain kata negasi penting
    cleaned_tokens = [
        word for word in tokens 
        if word not in string.punctuation and (word not in stop_words or word in important_words)
    ]
    # 4. Menggabungkan kembali token menjadi kalimat bersih
    return ' '.join(cleaned_tokens)

# Menerapkan fungsi prapemrosesan pada kolom teks
df['clean_text'] = df['text'].apply(preprocess_text)
df[['text', 'clean_text', 'sentiment']].head()

## Langkah 4: Pembagian Data Latih dan Data Uji (*Train-Test Split*)
Membagi dataset menjadi 80% data latih (*training*) dan 20% data uji (*testing*) menggunakan `stratify` untuk menjaga keseimbangan proporsi setiap kelas sentimen.

In [ ]:
# Membagi data feature (X) dan target label (y) dengan rasio 80:20
X_train, X_test, y_train, y_test = train_test_split(
    df['clean_text'], 
    df['sentiment'], 
    test_size=0.2, 
    random_state=42,
    stratify=df['sentiment']
)

print(f"Jumlah Data Latih (Train): {len(X_train)}")
print(f"Jumlah Data Uji (Test): {len(X_test)}")

## Langkah 5: Ekstraksi Fitur dengan TF-IDF Vectorizer
Mengubah teks mentah menjadi nilai numerik statistik menggunakan `TfidfVectorizer` dengan kombinasi Unigram dan Bigram `ngram_range=(1, 2)` agar frasa dua kata dapat terdeteksi.

In [ ]:
# Inisialisasi TfidfVectorizer dengan pembacaan kata tunggal dan pasangan dua kata (bigram)
vectorizer = TfidfVectorizer(ngram_range=(1, 2))

# Melakukan transformasi fit pada data latih dan pemetaan pada data uji
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

print("Dimensi matriks TF-IDF Data Latih:", X_train_vec.shape)

## Langkah 6: Pelatihan Model Klasifikasi (*Logistic Regression*)
Melatih model linear `LogisticRegression` menggunakan fitur numerik TF-IDF dari data latih.

In [ ]:
# Menginisialisasi dan melatih model Logistic Regression
model = LogisticRegression(max_iter=1000)
model.fit(X_train_vec, y_train)

print("Model Logistic Regression berhasil dilatih!")

## Langkah 7: Evaluasi Performa Model
Menguji akurasi dan kinerja klasifikasi model pada data uji menggunakan metrik *Precision*, *Recall*, *F1-Score*, dan *Accuracy*.

In [ ]:
# Melakukan prediksi label sentimen pada data uji
y_pred = model.predict(X_test_vec)

# Menampilkan skor akurasi dan laporan rincian evaluasi per kelas
print("Akurasi Model:", accuracy_score(y_test, y_pred))
print("\nLaporan Klasifikasi:")
print(classification_report(y_test, y_pred, zero_division=0))

## Langkah 8: Pengujian Prediksi pada Kalimat Ulasan Baru
Menguji model yang telah dilatih menggunakan sampel kalimat ulasan baru di luar dataset latih.

In [ ]:
# Daftar kalimat ulasan baru yang akan diprediksi
kalimat_baru = [
    "Barang sangat memuaskan, pengiriman super cepat!",
    "Pelayanan toko ini sangat buruk dan mengecewakan.",
    "Biasa saja, tidak ada yang spesial dari produk ini.",
    "Barangnya tidak bagus sama sekali, saya sangat kecewa.",
    "Meskipun agak mahal, kualitasnya tidak mengecewakan."
]

# Melakukan prapemrosesan dan ekstraksi fitur TF-IDF pada data baru
clean_new = [preprocess_text(k) for k in kalimat_baru]
new_vec = vectorizer.transform(clean_new)

# Melakukan prediksi sentimen
prediksi = model.predict(new_vec)

# Menampilkan hasil prediksi
for teks, pred in zip(kalimat_baru, prediksi):
    print(f"Ulasan: \"{teks}\"\nHasil Prediksi: {pred.upper()}\n")